<a href="https://colab.research.google.com/github/elhamod/IS883_Fall_2026/blob/main/Session%2005/IS883_Session05_MCP_Extras.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# IS883 Session 5 (optional extra): Agents That Act

In class, Gemini filed a GitHub issue through GitHub's own MCP server. This optional notebook does the same with three more real servers, then shows where to find more.

1. **Email with Resend**: the agent sends you an email.
2. **Gmail and Google Calendar through Zapier**: the agent emails from your Gmail and adds an event to your calendar.
3. **Linear**: the agent creates a task on a project board.
4. **Finding MCP servers**: where to look, and when MCP is *not* the right tool.

Every part is independent: do the ones you like. All are free. Each needs one key, which goes in Colab Secrets, never in the notebook.

# Part 0: Setup

In [ ]:
# Install Google's GenAI SDK for the Gemini API (usually already there in Colab).
!pip install -q google-genai

**Where the API key comes from.** Never paste a key into a notebook — anyone you share the file with gets your key, and a key pushed to a public repo is found within minutes. Instead the helper below looks for it in three places, in order:

1. **Colab Secrets** (the 🔑 panel on the left) — this is how you should do it in this course.
2. An **environment variable** of the same name — for VS Code or a local Jupyter.
3. Failing both, it **asks you to paste it**, and even then the value never lands in the file.

In [ ]:
# Read the API key from wherever this notebook is running, without writing it into the file.
import os, getpass

def get_key(name):
    try:
        from google.colab import userdata
        return userdata.get(name)
    except Exception:
        return os.environ.get(name) or getpass.getpass(f"{name}: ")

In [ ]:
# Connect to the Gemini API using the key you saved as a Colab secret (named MyGeminiKey).
from google import genai
from google.genai import types

client = genai.Client(api_key=get_key('MyGeminiKey'))

MODEL = "gemini-3.1-flash-lite"

All three servers below are reached the same way as GitHub's in class: we pass the server's address and our key to `client.interactions.create`, and Google's servers do the connecting. The helper below prints what happened.

In [ ]:
# Helper: print each step Google's servers took (tool calls and results) and the token bill for the whole run.
def show_steps(interaction):
    for step in interaction.steps or []:
        if step.type == "function_call":                  # Gemini asks the MCP server to run a tool
            print("Gemini called:", step.name, "with", step.arguments)
        elif step.type == "function_result":              # what the MCP server sent back
            print("Server returned:", str(getattr(step.result, "result", step.result))[:300])
    if not any(step.type == "function_call" for step in interaction.steps or []):
        print("Gemini did not call any tool this time.")
    usage = interaction.usage
    print("*" * 20)
    print("input_tokens:   ", usage.total_input_tokens)
    print("output_tokens:  ", usage.total_output_tokens)
    print("thinking_tokens:", usage.total_thought_tokens)
    print("tools_tokens:   ", usage.total_tool_use_tokens)
    print("total_tokens:   ", usage.total_tokens)
    print("*" * 20)

# Part 1: Send an Email with Resend

[Resend](https://resend.com) is an email service for developers, and it runs an official MCP server. On the free plan, until you verify a domain of your own, it can only send email **to the address you signed up with**. That is a handy safety rail: the agent can't email anyone else.

**Setup (about 3 minutes):**
1. Sign up at [resend.com](https://resend.com) (free).
2. Open **API Keys → Create API Key**, give it *Sending access*, and copy it.
3. Save it in Colab Secrets as **`MyResendKey`** (Notebook access on).

In [ ]:
# The address you signed up to Resend with: the only address the free plan can send to.
MY_EMAIL = "you@example.com"

In [ ]:
# Ask Gemini to write and send the email through Resend's MCP server.
task = (f"Send an email to {MY_EMAIL} from onboarding@resend.dev with the subject 'Hello from my agent'. "
        "In the body, write two sentences explaining what an MCP server is, for a business audience.")

interaction = client.interactions.create(
    model=MODEL,
    input=task,
    tools=[{
        "type": "mcp_server",
        "name": "resend",                                   # any label, no dashes
        "url": "https://mcp.resend.com/mcp",
        "headers": {"Authorization": "Bearer " + get_key("MyResendKey")},
    }],
    store=False,
)
print(interaction.output_text)

In [ ]:
# Peek behind the scenes: the tools Gemini called, what came back, and the token bill.
show_steps(interaction)

Check your inbox (and the spam folder): the email is there.

**Question 1.** Look at the list of tools the server offered. Which of them could cause real harm if a prompt (or a document the agent read) told it to use them? What does the free plan's "only to your own address" rule protect you from?

**Answer**

*Provide your answer here*

# Part 2: Gmail and Google Calendar through Zapier

Google's own MCP servers for Gmail and Calendar exist, but they are still a developer preview that needs a Google Cloud project. [Zapier](https://zapier.com) offers a shortcut: you connect your Google account to Zapier once, choose which actions the agent may use, and Zapier gives you one MCP server for all of them.

**Free plan:** Zapier MCP is included. The free plan has 100 *tasks* a month and each action the agent takes uses 2, so about **50 actions a month**.

**Use a personal Gmail account**, not your BU account: BU-managed Google accounts may block third-party apps.

**Setup (about 8–10 minutes):**
1. Sign up at [zapier.com](https://zapier.com) (free; *Sign up with Google* is quickest).
2. Go to [mcp.zapier.com](https://mcp.zapier.com), click **Add MCP server**, and under *Choose your AI agent* pick **Other**.
3. Add two tools: **Gmail → Send Email** and **Google Calendar → Create Detailed Event**. Each one asks you to sign in to Google once.
4. Open the **Connect** tab and click **Generate token**. Copy it right away: it is shown only once.
5. Save it in Colab Secrets as **`MyZapierToken`** (Notebook access on). Treat it like a password: it controls your Gmail. If it leaks, click **Rotate token**.

In [ ]:
# Zapier's MCP server is one address for every action you enabled; the token says which account is yours.
ZAPIER = {
    "type": "mcp_server",
    "name": "zapier",
    "url": "https://mcp.zapier.com/api/v1/connect",
    "headers": {"Authorization": "Bearer " + get_key("MyZapierToken")},
}
MY_GMAIL = "you@gmail.com"

In [ ]:
# Ask Gemini to send an email from your Gmail, to yourself.
interaction = client.interactions.create(
    model=MODEL,
    input=f"Send an email to {MY_GMAIL} with the subject 'Agent test' and a one-line body saying the Zapier MCP server works.",
    tools=[ZAPIER], store=False)
print(interaction.output_text)
show_steps(interaction)

In [ ]:
# Ask Gemini to put an event on your Google Calendar.
interaction = client.interactions.create(
    model=MODEL,
    input="Create a Google Calendar event called 'IS883 project work session' next Wednesday from 6 to 7 PM Eastern time.",
    tools=[ZAPIER], store=False)
print(interaction.output_text)

In [ ]:
# Peek behind the scenes: the tools Gemini called, what came back, and the token bill.
show_steps(interaction)

**Question 2.** Gemini had to work out "next Wednesday" by itself. Did it get the date right? What does that tell you about letting an agent act on your calendar without a confirmation step, and where in a real app would you add one?

**Answer**

*Provide your answer here*

# Part 3: Create a Task in Linear

[Linear](https://linear.app) is a project-tracking tool popular with product teams, and it runs an official MCP server. It is a good picture of what agents do inside companies: turn a conversation into tracked work.

**Setup (about 5 minutes):**
1. Sign up at [linear.app](https://linear.app) (free plan) and create a workspace with one team.
2. Open **Settings → Security & access → Personal API keys**, create a key and copy it.
3. Save it in Colab Secrets as **`MyLinearKey`** (Notebook access on).

In [ ]:
# Ask Gemini to create a task in your Linear workspace.
interaction = client.interactions.create(
    model=MODEL,
    input=("Create an issue in my Linear team titled 'Draft the project checkpoint README' with a one-paragraph "
           "description of what the checkpoint should contain for a GenAI class project. Tell me the link to the new issue."),
    tools=[{
        "type": "mcp_server",
        "name": "linear",                                   # any label, no dashes
        "url": "https://mcp.linear.app/mcp",
        "headers": {"Authorization": "Bearer " + get_key("MyLinearKey")},
    }],
    store=False,
)
print(interaction.output_text)

In [ ]:
# Peek behind the scenes: the tools Gemini called, what came back, and the token bill.
show_steps(interaction)

# Part 4: Finding MCP Servers

MCP shines when your app needs a capability **someone else maintains**: filings, search, a calendar, a ticketing system. Many companies now run official servers, and most have a **free** option; some need a free API key or a sign-in.

**A word of caution first.** Inside a Streamlit app, calling a Python library or API directly is often simpler, faster and easier to debug than running MCP. Use MCP when it saves you real work, not to look modern.

**Where to find servers:** the official [MCP Registry](https://registry.modelcontextprotocol.io), [GitHub's MCP page](https://github.com/mcp), [mcp.so](https://mcp.so) and [Smithery](https://smithery.ai). Anyone can publish to these, so being listed is not a sign of quality: prefer servers run by the company that owns the data (GitHub, Resend, Linear, Zapier).

**Should your team build its own MCP server?** Usually not. Write a normal Python function first; turn it into a server only when a **second** app or person needs the same tool.

**Question 3 (optional).** Browse the directories above and find one MCP server that could help your team's project. What would your app ask it to do, what key or sign-in would each user need, and what is the plain-Python alternative? Which would you choose, and why?

**Answer**

*Provide your answer here*